# VLA Failure Visualization
Use this notebook to qualitatively inspect the top failure cases identified by `analyze_vla_failures.py`.

This tool supports both **Single-Frame** and **Episodic** artifacts.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output

# Update RESULTS_BASE to your specific experiment directory
RESULTS_DIR = "results/vla_failure_analysis/bridge_quest_32_888_4_vanilla_episodic"

RESULTS_BASE = f"{RESULTS_DIR}/worst"


def get_failure_folders(root):
    if not os.path.exists(root):
        return []
    folders = [
        f
        for f in os.listdir(root)
        if os.path.isdir(os.path.join(root, f)) and f.startswith("failure_")
    ]
    return sorted(folders)


folders = get_failure_folders(RESULTS_BASE)
if not folders:
    print(f"No failure folders found in {RESULTS_BASE}. Please check the path.")

In [2]:
def show_failure(folder_name, step_idx=0):
    path = os.path.join(RESULTS_BASE, folder_name)
    data_path = os.path.join(path, "data.npz")

    if not os.path.exists(data_path):
        print(f"Data not found at {data_path}")
        return

    data = np.load(data_path, allow_pickle=True)
    is_episodic = "images" in data.files

    instruction = data["instruction"]
    if is_episodic:
        images = data["images"]
        gt_actions = data["gt_actions"]
        pred_actions = data["pred_actions"]
        gt_tokens = data["gt_tokens"]
        pred_tokens = data["pred_tokens"]
        l1_errors = data["l1_errors"]
        mean_err = data["mean_l1_error"]

        T = images.shape[0]
        step_idx = min(step_idx, T - 1)

        current_img = images[step_idx]
        current_gt = gt_actions[step_idx]
        current_pred = pred_actions[step_idx]
        current_gt_tok = gt_tokens[step_idx]
        current_pred_tok = pred_tokens[step_idx]
        current_err = l1_errors[step_idx]

        clear_output(wait=True)
        print(f"Mode: EPISODIC | Failure: {folder_name} | Step {step_idx}/{T - 1}")
        print(f"Instruction: {instruction}")
        print(f"Step L1 Error: {current_err:.4f} (Episode Mean: {mean_err:.4f})")
    else:
        current_img = data["image"]
        current_gt = data["gt_action"]
        current_pred = data["pred_action"]
        current_gt_tok = data["gt_tokens"]
        current_pred_tok = data["pred_tokens"]
        current_err = data["l1_error"]

        clear_output(wait=True)
        print(f"Mode: SINGLE-FRAME | Failure: {folder_name}")
        print(f"Instruction: {instruction}")
        print(f"L1 Error: {current_err:.4f}")

    # Visualization
    fig, axes = plt.subplots(
        1, 3 if is_episodic else 2, figsize=(20 if is_episodic else 14, 6)
    )

    # 1. Observation
    axes[0].imshow(current_img)
    axes[0].set_title(
        f"Observation (Step {step_idx})" if is_episodic else "Observation"
    )
    axes[0].axis("off")

    # 2. Action Comparison
    dims = ["x", "y", "z", "rx", "ry", "rz", "gripper"]
    x = np.arange(len(dims))
    axes[1].bar(x - 0.2, current_gt, 0.4, label="GT", color="tab:blue")
    axes[1].bar(x + 0.2, current_pred, 0.4, label="Pred", color="tab:orange")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(dims)
    axes[1].set_title("Action Comparison")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    if is_episodic:
        # 3. Error over time
        axes[2].plot(l1_errors, color="tab:red", alpha=0.3)
        axes[2].scatter([step_idx], [current_err], color="red", zorder=5)
        axes[2].set_title("L1 Error Trend")
        axes[2].set_xlabel("Step")
        axes[2].set_ylabel("L1")
        axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Token and Action Tables
    print("\n--- Token Comparison ---")
    print(f"{'Idx':<5} | {'Ground Truth':<15} | {'Predicted':<15} | {'Match'}")
    print("-" * 45)
    for i in range(len(current_gt_tok)):
        match = "OK" if current_gt_tok[i] == current_pred_tok[i] else "DIFF"
        print(f"{i:<5} | {current_gt_tok[i]:<15} | {current_pred_tok[i]:<15} | {match}")

    print("\n--- Action Value Table ---")
    print(f"{'Dim':<8} | {'Ground Truth':<12} | {'Predicted':<12} | {'Diff'}")
    print("-" * 45)
    for i, dim in enumerate(dims):
        diff = current_gt[i] - current_pred[i]
        print(
            f"{dim:<8} | {current_gt[i]:>12.4f} | {current_pred[i]:>12.4f} | {diff:>8.4f}"
        )


def on_folder_change(change):
    if change["type"] == "change" and change["name"] == "value":
        folder = change["new"]
        if not folder:
            return
        data = np.load(
            os.path.join(RESULTS_BASE, folder, "data.npz"), allow_pickle=True
        )
        if "images" in data.files:
            T = data["images"].shape[0]
            step_slider.max = T - 1
            step_slider.value = 0
            step_slider.disabled = False
        else:
            step_slider.max = 0
            step_slider.value = 0
            step_slider.disabled = True


folder_dropdown = widgets.Dropdown(
    options=folders,
    value=folders[0] if folders else None,
    description="Folder:",
    layout={"width": "max-content"},
)
step_slider = widgets.IntSlider(
    value=0, min=0, max=0, description="Step:", disabled=True
)

folder_dropdown.observe(on_folder_change, names="value")
if folders:
    on_folder_change({"type": "change", "name": "value", "new": folder_dropdown.value})

ui = widgets.VBox([folder_dropdown, step_slider])
out = widgets.interactive_output(
    show_failure, {"folder_name": folder_dropdown, "step_idx": step_slider}
)
display(ui, out)

Output()

In [4]:
def plot_every_tenth_frame(folder_name, save_path=None):
    """Plot every 10th frame in a grid for episodic data"""
    path = os.path.join(RESULTS_BASE, folder_name)
    data_path = os.path.join(path, "data.npz")

    if not os.path.exists(data_path):
        print(f"Data not found at {data_path}")
        return

    data = np.load(data_path, allow_pickle=True)

    if "images" not in data.files:
        print(f"Not episodic data, skipping {folder_name}")
        return

    images = data["images"]
    l1_errors = data["l1_errors"]
    pred_actions = data["pred_actions"]
    instruction = data["instruction"]

    T = images.shape[0]
    indices = list(range(0, T, 10))

    n_frames = len(indices)
    ncols = min(4, n_frames)
    nrows = (n_frames + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for i, idx in enumerate(indices):
        gripper_val = pred_actions[idx][-1]  # Last dim is gripper
        gripper_state = "OPEN" if gripper_val < 0 else "CLOSED"

        axes[i].imshow(images[idx])
        axes[i].set_title(
            f"Step {idx} | Predicted gripper state: {gripper_state}\nDecoded action L1: {l1_errors[idx]:.3f} error"
        )
        axes[i].axis("off")

    # Hide unused subplots
    for i in range(n_frames, len(axes)):
        axes[i].axis("off")

    fig.suptitle(f"example ID: {folder_name}\nInstruction: {instruction}", fontsize=12)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved to {save_path}")

    plt.show()


os.makedirs(f"{RESULTS_DIR}/analysis", exist_ok=True)
# Plot all episodic failures
for folder in folders:
    plot_every_tenth_frame(folder, save_path=f"{RESULTS_DIR}/analysis/{folder}.png")

Not episodic data, skipping failure_001_err0.6319
Not episodic data, skipping failure_002_err0.6299
Not episodic data, skipping failure_003_err0.5975
Not episodic data, skipping failure_004_err0.5743
Not episodic data, skipping failure_005_err0.5605
Not episodic data, skipping failure_006_err0.5569
Not episodic data, skipping failure_007_err0.5565
Not episodic data, skipping failure_008_err0.5406
Not episodic data, skipping failure_009_err0.5345
Not episodic data, skipping failure_010_err0.5043
